# 02 — E2 AdminQA Smoke Test (VNAB Pilot v1.1)
**Mục đích:** kiểm tra tương thích dữ liệu địa giới và checkpoint MRC đã dùng ở E1. Không fine-tune, không cập nhật trọng số, không đọc test.

Chạy lần lượt 01 → 07. Upload đúng `vnab-pilot-v1.1.zip` ở mục 02. CPU đủ cho smoke test; ưu tiên GPU nếu đã được cấp. Giữ nguyên notebook E1. Mục 08 chỉ chạy sau khi pilot hoàn thành, để kiểm tra tokenization của v1.0 (không huấn luyện).

Dữ liệu nguồn giữ nguyên Unicode và offset. Metric pilot giữ quy tắc E1 (NFC, lowercase, whitespace, giữ dấu câu). 4 dev chỉ để kiểm tra kỹ thuật, không phải kết quả chính thức.

**Chạy Jupyter trên máy.** Mở notebook từ thư mục gốc của repo hoặc từ `research/`. Notebook tự tìm ZIP trong `research/handoff/` và lưu kết quả vào `research/data/local/`. Trên Windows, dùng đường dẫn cache Hugging Face ngắn và không dấu để SentencePiece nạp được tệp mô hình. Trên Colab, tải ZIP dữ liệu lên và tải ZIP kết quả về theo hướng dẫn từng mục.


## 01. Môi trường
Giữ phiên bản E1 nếu đã cài. Nếu cell báo cần cài, xem log, sau khi cài xong hãy Restart session và chạy lại 01. Không cài lại torch.

In [ ]:
import sys, subprocess
from importlib.metadata import version, PackageNotFoundError

REQUIRED = {
    "transformers": "4.57.1",
    "datasets": "4.8.5",
    "huggingface-hub": "0.36.2",
    "accelerate": "1.15.0",
    "sentencepiece": "0.2.2",
}
def installed(name):
    try:
        return version(name)
    except PackageNotFoundError:
        return None

changes = [f"{k}=={v}" for k,v in REQUIRED.items() if installed(k) != v]
changes += [k for k in ("pandas", "jsonschema", "protobuf") if installed(k) is None]
if changes:
    print("Cần cài:", changes, flush=True)
    subprocess.check_call([sys.executable, "-m", "pip", "install", *changes])
    raise SystemExit("Cài xong. Restart session rồi chạy lại mục 01.")

import torch, transformers, pandas as pd, json, zipfile, hashlib, io, re, unicodedata
import jsonschema
from collections import Counter
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForQuestionAnswering, pipeline, set_seed
from huggingface_hub import HfApi

SEED = 42
set_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA check:", (torch.ones(1,device=DEVICE)*2).item())
for k in REQUIRED:
    print(k, installed(k))
print("torch", torch.__version__)
print("ENVIRONMENT OK")

## 02. Upload và kiểm tra file ZIP
Không giải nén lên thư mục dự án. Kiểm tra checksum, schema, span và overlap train/dev ngay trong ZIP. Không cần upload tập test.

In [ ]:
try:
    from google.colab import files
except ImportError:
    files = None

research_dir = next(
    (p for p in (Path("research"), Path.cwd()) if (p / "handoff").is_dir()),
    None,
)

def upload_zip(expected):
    if research_dir is not None:
        matches = list((research_dir / "handoff").glob(f"*/{expected}"))
        if len(matches) == 1:
            return matches[0]
        if len(matches) > 1:
            raise ValueError(f"Có nhiều ZIP tên {expected} trong thư mục bàn giao.")
    p = Path(expected)
    if not p.exists():
        if files is None:
            raise FileNotFoundError(f"Không tìm thấy {expected} trong repo hoặc thư mục hiện tại.")
        uploaded = files.upload()
        if expected not in uploaded:
            raise ValueError(f"Hãy upload đúng file {expected}")
        p.write_bytes(uploaded[expected])
    return p

def load_package(path, expected_counts):
    out = {}
    with zipfile.ZipFile(path) as z:
        checks = z.read("checksums.sha256").decode("utf-8").splitlines()
        for line in checks:
            sha, name = line.split(maxsplit=1)
            assert hashlib.sha256(z.read(name)).hexdigest() == sha, f"Checksum sai: {name}"
        for kind in ["qa", "mrc"]:
            schema = json.loads(z.read(f"{kind}.schema.json"))
            validator = jsonschema.Draft202012Validator(schema)
            out[kind] = {}
            for split in ["train", "dev"]:
                name = f"{kind}_{split}.jsonl"
                records = [json.loads(s) for s in z.read(name).decode("utf-8-sig").splitlines() if s.strip()]
                assert len(records) == expected_counts[kind][split], name
                for row in records:
                    validator.validate(row)
                    assert row["review_status"] == "verified", row["id"]
                out[kind][split] = records
            a,b = out[kind]["train"],out[kind]["dev"]
            for split, recs in out[kind].items():
                assert len({r["id"] for r in recs}) == len(recs), (kind,split,"duplicate ID")
            for key in ["id","group_id","context","question"]:
                assert not ({r[key] for r in a} & {r[key] for r in b}), (kind,key)
        for key in ["mapping_ids","old_unit_ids","new_unit_ids"]:
            a,b = out["qa"]["train"],out["qa"]["dev"]
            assert not ({x for r in a for x in r[key]} & {x for r in b for x in r[key]}), key
        for split in ["train","dev"]:
            lookup = {r["id"]:r for r in out["qa"][split]}
            for row in out["mrc"][split]:
                texts, starts = row["answers"]["text"],row["answers"]["answer_start"]
                assert len(texts) == len(starts) > 0
                for text,start in zip(texts,starts):
                    assert 0 <= start and row["context"][start:start+len(text)] == text, row["id"]
                q = lookup[row["id"]]
                for key in ["question","context","group_id","province","type","source_doc","article_clause","context_origin","answerable","review_status"]:
                    assert row[key] == q[key], (row["id"],key)
                assert (q["answer_text"],q["answer_start"]) in list(zip(texts,starts))
        manifest = pd.read_csv(io.BytesIO(z.read("split_manifest.csv")),dtype=str)
        for split in ["train","dev"]:
            assert set(manifest.loc[manifest["split"]==split,"id"]) == {r["id"] for r in out["qa"][split]}, split
    return out

PILOT_PATH = upload_zip("vnab-pilot-v1.1.zip")
pilot = load_package(PILOT_PATH, {"qa":{"train":16,"dev":4},"mrc":{"train":16,"dev":4}})
print("CHECKSUM / SCHEMA / SPAN / TRAIN-DEV OVERLAP: OK")
for split, recs in pilot["mrc"].items():
    print(split, len(recs), "MRC |", "non-NFC:",sum(not unicodedata.is_normalized("NFC",r["context"]) for r in recs))
print("Source context kept unchanged.")

## 03. Load đúng checkpoint E1
Dùng revision cố định từ config.json của E1, không tự đổi model. Không đăng nhập nếu tải công khai thành công.

In [ ]:
MODEL_NAME = "deepset/xlm-roberta-base-squad2"
MODEL_REVISION = "a5fab9908c8d856e8c583fd41ba6d92444e46477"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, revision=MODEL_REVISION, use_fast=True)
model, loading_info = AutoModelForQuestionAnswering.from_pretrained(
    MODEL_NAME, revision=MODEL_REVISION, output_loading_info=True
)
assert tokenizer.is_fast
assert not loading_info["missing_keys"] and not loading_info["mismatched_keys"], loading_info
model.to(DEVICE)
model.eval()
print("Model:", MODEL_NAME)
print("Revision:", MODEL_REVISION)
print("Tokenizer:", type(tokenizer).__name__)
print("Fast tokenizer:", tokenizer.is_fast)
print("Missing keys:", loading_info["missing_keys"])
print("Mismatched keys:", loading_info["mismatched_keys"])
print("Parameters:", sum(p.numel() for p in model.parameters()))
print("MODEL LOADED SUCCESSFULLY")

## 04. Tokenize và chuyển answer_start sang nhãn token
Giữ nguyên chuỗi và offset trong ZIP. Tạo bản NFC cho mô hình, tính lại hai đầu span từ độ dài tiền tố NFC và kiểm lại bằng cách cắt chuỗi. Tokenizer nhận cặp question/context đã chuẩn hóa. `sequence_ids == 1` nhận diện context. `offset_mapping` ánh xạ token về chỉ số ký tự trong bản NFC. Nhãn end_positions là vị trí token cuối (inclusive), khác với end ký tự (exclusive).

Dùng max length 384 như E1. Với các đoạn ngắn, không cắt ngữ cảnh. Nếu có mẫu vượt 384 hoặc không tìm được nhãn hợp lệ, dừng để sửa preprocessing (không tự gán CLS). Cảnh báo span token rộng hơn gold được lưu riêng; khác biệt do khoảng trắng/NFC được coi là tương đương.

In [ ]:
MAX_LENGTH = 384

def norm_boundary(s):
    return unicodedata.normalize("NFC", s).strip()

def make_feature(row, split):
    raw_context = row["context"]
    ctx = unicodedata.normalize("NFC", raw_context)
    question = unicodedata.normalize("NFC", row["question"])
    encoded = tokenizer(question, ctx, truncation=False, return_offsets_mapping=True)
    offsets = encoded["offset_mapping"]
    seq = encoded.sequence_ids()
    if len(encoded["input_ids"]) > MAX_LENGTH:
        raise ValueError(f"{row['id']}: > {MAX_LENGTH} tokens; cần chia cửa sổ, không truncate âm thầm.")
    valid = [i for i,(a,b) in enumerate(offsets) if seq[i]==1 and b>a]
    candidates = []
    for raw_text, raw_start in zip(row["answers"]["text"], row["answers"]["answer_start"]):
        raw_end = raw_start + len(raw_text)
        start = len(unicodedata.normalize("NFC", raw_context[:raw_start]))
        end = len(unicodedata.normalize("NFC", raw_context[:raw_end]))
        text = unicodedata.normalize("NFC", raw_text)
        if ctx[start:end] != text:
            raise ValueError(f"{row['id']}: span không khớp sau khi quy đổi sang NFC.")
        # Chọn các token context phủ ký tự đầu và cuối của gold span.
        left = [i for i in valid if offsets[i][0] <= start < offsets[i][1]]
        right = [i for i in valid if offsets[i][0] < end <= offsets[i][1]]
        for si in left:
            for ei in right:
                if si > ei:
                    continue
                a,b = offsets[si][0], offsets[ei][1]
                if not (a <= start < end <= b):
                    continue
                extracted = ctx[a:b]
                match = norm_boundary(extracted) == norm_boundary(text)
                candidates.append((not match, (start-a)+(b-end), si, ei, extracted, raw_text, raw_start, raw_end, start, end))
    if not candidates:
        raise ValueError(f"{row['id']}: không tìm được token span hợp lệ.")
    mismatch, _, si, ei, extracted, gold, raw_start, raw_end, start, end = min(candidates)
    feature = {k:encoded[k] for k in tokenizer.model_input_names if k in encoded}
    feature.update(start_positions=si,end_positions=ei)
    audit = {
        "id":row["id"], "split":split, "province":row["province"], "tokens":len(encoded["input_ids"]),
        "gold":gold, "answer_start":raw_start, "answer_end_exclusive":raw_end,
        "model_answer_start_nfc":start, "model_answer_end_nfc_exclusive":end,
        "start_token":si, "end_token_inclusive":ei,
        "token_span_text":extracted, "boundary_match_nfc_trim":not mismatch
    }
    return feature,audit

def tokenize_package(package):
    feats, audits = {}, []
    for split, records in package["mrc"].items():
        feats[split] = []
        for row in records:
            feat,audit = make_feature(row,split)
            feats[split].append(feat)
            audits.append(audit)
    return feats,pd.DataFrame(audits)

features, token_audit = tokenize_package(pilot)
display(token_audit[["split","tokens","gold","token_span_text","start_token","end_token_inclusive","boundary_match_nfc_trim"]])
print("Tokenized:", len(token_audit))
print("Max tokens:", token_audit["tokens"].max())
print("Boundary mismatches:", int((~token_audit["boundary_match_nfc_trim"]).sum()))
print("TOKENIZATION OK — xem cảnh báo boundary nếu có")

## 05. Smoke test batch và loss
Padding động trong từng batch. Start/end labels là chỉ số token. Chạy tất cả 16 train và 4 dev theo batch 4; dùng `inference_mode()` và `model.eval()`, không backward/optimizer, không cập nhật trọng số. Loss hữu hạn xác nhận mô hình tiếp nhận dữ liệu và nhãn (không đánh giá chất lượng học).

In [ ]:
import math
BATCH_SIZE = 4

def check_batches(feats):
    logs = []
    model.eval()
    for split, records in feats.items():
        for i in range(0,len(records),BATCH_SIZE):
            chunk = records[i:i+BATCH_SIZE]
            inputs = tokenizer.pad(
                [{k:v for k,v in r.items() if k not in ["start_positions","end_positions"]} for r in chunk],
                padding=True, return_tensors="pt"
            )
            inputs["start_positions"] = torch.tensor([r["start_positions"] for r in chunk])
            inputs["end_positions"] = torch.tensor([r["end_positions"] for r in chunk])
            inputs = {k:v.to(DEVICE) for k,v in inputs.items()}
            with torch.inference_mode():
                output = model(**inputs)
            loss = float(output.loss.item())
            assert math.isfinite(loss), (split,i,"invalid loss")
            assert output.start_logits.shape == inputs["input_ids"].shape
            assert output.end_logits.shape == inputs["input_ids"].shape
            logs.append({"split":split,"batch":i//BATCH_SIZE+1,"samples":len(chunk),
                         "shape":list(inputs["input_ids"].shape),"loss":loss})
    return pd.DataFrame(logs)

batch_log = check_batches(features)
display(batch_log)
print("BATCH / FORWARD / LOSS OK — không huấn luyện")

## 06. Inference trên 4 dev
Cùng checkpoint và cấu hình inference như E1. EM/F1 theo quy tắc E1. Không xem 4 dev này là test độc lập; điểm không phải tiêu chí pass/fail smoke test.

In [ ]:
import time
QA_SETTINGS = {"max_seq_len":384,"doc_stride":128,"max_answer_len":128,"handle_impossible_answer":False}
qa = pipeline("question-answering", model=model, tokenizer=tokenizer, device=0 if DEVICE.type=="cuda" else -1)

def normalize_answer(s):
    return " ".join(unicodedata.normalize("NFC",s).lower().split())

def score_one(pred,gold):
    p,g = normalize_answer(pred),normalize_answer(gold)
    pt,gt = p.split(),g.split()
    em = int(p==g)
    if not pt or not gt:
        return em,float(pt==gt)
    common = sum((Counter(pt)&Counter(gt)).values())
    return em, 2*common/(len(pt)+len(gt))

predictions = []
for row in pilot["mrc"]["dev"]:
    tic = time.perf_counter()
    result = qa(
        question=unicodedata.normalize("NFC", row["question"]),
        context=unicodedata.normalize("NFC", row["context"]),
        **QA_SETTINGS,
    )
    if DEVICE.type=="cuda":
        torch.cuda.synchronize()
    ms = (time.perf_counter()-tic)*1000
    pred = result["answer"]
    scores = [score_one(pred,g) for g in row["answers"]["text"]]
    predictions.append({
        "id":row["id"],"group_id":row["group_id"],"province":row["province"],"type":row["type"],
        "source_doc":row["source_doc"],"article_clause":row["article_clause"],
        "question":row["question"],"gold":row["answers"]["text"][0],"prediction":pred,
        "score":float(result["score"]),"pred_start_nfc":result["start"],"pred_end_nfc_exclusive":result["end"],
        "exact_match":max(s[0] for s in scores),"f1":max(s[1] for s in scores),"latency_ms":ms})
pred_df = pd.DataFrame(predictions)
display(pred_df[["province","question","gold","prediction","exact_match","f1"]])
metrics = {"samples":len(pred_df),"EM_percent":float(pred_df.exact_match.mean()*100),
           "F1_percent":float(pred_df.f1.mean()*100),"avg_latency_ms":float(pred_df.latency_ms.mean()),
           "device":str(DEVICE),"scope":"T1, pilot dev only, not independent test"}
print(json.dumps(metrics,ensure_ascii=False,indent=2))

## 07. Lưu báo cáo smoke test
Tải ZIP kết quả và gửi Vi. PASS kỹ thuật không có nghĩa mô hình trả lời đúng; nếu boundary mismatch > 0 thì cần rà trước fine-tune. Không có checkpoint mới vì chưa huấn luyện.

In [ ]:
import shutil
results_dir = research_dir / "data" / "local" if research_dir is not None else Path.cwd()
out = results_dir / "E2_smoke_results"
out.mkdir(parents=True, exist_ok=True)
token_audit.to_csv(out/"token_audit.csv",index=False,encoding="utf-8-sig")
batch_log.to_csv(out/"batch_log.csv",index=False,encoding="utf-8-sig")
pred_df.to_csv(out/"predictions_dev.csv",index=False,encoding="utf-8-sig")

boundary_issues = int((~token_audit["boundary_match_nfc_trim"]).sum())
report = {
    "status":"PASS" if boundary_issues==0 else "PASS_WITH_BOUNDARY_WARNINGS",
    "dataset":"vnab-pilot-v1.1","zip_sha256":hashlib.sha256(PILOT_PATH.read_bytes()).hexdigest(),
    "train_samples":16,"dev_samples":4,"tokenized":len(token_audit),"batch_samples":int(batch_log.samples.sum()),
    "boundary_mismatches":boundary_issues,"model":MODEL_NAME,"revision":MODEL_REVISION,
    "tokenizer":type(tokenizer).__name__,"fast_tokenizer":tokenizer.is_fast,
    "missing_keys":loading_info["missing_keys"],"mismatched_keys":loading_info["mismatched_keys"],
    "model_parameters":sum(p.numel() for p in model.parameters()),
    "fine_tuned_by_team":False,"device":str(DEVICE),"gpu":torch.cuda.get_device_name(0) if DEVICE.type=="cuda" else None,
    "seed":SEED,"max_length":MAX_LENGTH,"batch_size":BATCH_SIZE,"qa_settings":QA_SETTINGS,
    "unicode_policy":"source ZIP text and offsets unchanged; tokenize and infer on NFC with validated span remapping",
    "metric_normalization":"NFC, lowercase, whitespace; retain punctuation and Vietnamese accents",
    "libraries":{**{k:installed(k) for k in REQUIRED},"torch":str(torch.__version__)},
    "metrics":metrics
}
(out/"smoke_report.json").write_text(json.dumps(report,ensure_ascii=False,indent=2),encoding="utf-8")
archive = shutil.make_archive(str(out), "zip", root_dir=out)
print("SMOKE TEST:",report["status"])
print("Saved:",archive)
if files is not None:
    files.download(archive)

## 08. Tùy chọn: Kiểm tra dataset v1.0 sau khi pilot PASS
Chỉ chạy sau khi xem báo cáo mục 07. Upload `vnab-dataset-v1.0.zip` riêng. Kiểm tra toàn bộ 163 MRC, xác nhận 20 mẫu pilot không thay đổi. Chưa train; chưa có test. Nếu có lỗi, gửi output trước khi chuyển notebook fine-tuning.

In [ ]:
assert report["status"]=="PASS", "Rà lỗi boundary trong pilot trước."
FULL_PATH = upload_zip("vnab-dataset-v1.0.zip")
full = load_package(FULL_PATH,{"qa":{"train":199,"dev":50},"mrc":{"train":126,"dev":37}})
for kind in ["qa","mrc"]:
    for split in ["train","dev"]:
        indexed = {r["id"]:r for r in full[kind][split]}
        assert all(indexed.get(r["id"])==r for r in pilot[kind][split]), (kind,split)
full_features,full_token_audit = tokenize_package(full)
full_audit_path = results_dir / "E2_v1_token_audit.csv"
full_token_audit.to_csv(full_audit_path, index=False, encoding="utf-8-sig")
print("V1.0 PILOT PRESERVATION OK")
print("Train/dev tokenized:",len(full_features["train"]),len(full_features["dev"]))
print("Boundary mismatches:",int((~full_token_audit.boundary_match_nfc_trim).sum()))
print("Max tokens:",full_token_audit.tokens.max())
print("Saved:", full_audit_path)
if files is not None:
    files.download(str(full_audit_path))